# DG advection of a cosine in an interval

Donea, J. & Huerta, A. (2003). *Finite Element Methods for Flow Problems*. $\S 3.11.1$


$$
\mathbb{S}
\begin{cases}
\Omega = [0, L_x] \\
u_0(x)=\begin{cases}
\frac{1}{2}(1+\cos(\pi(x-x_0)/\sigma)) & \text{if}~|x-x_0|\leq\tfrac{\sigma}{2} \\
0 & \text{otherwise}
\end{cases} \\
u_{\text{I}}(x=0) = 0 \\
\textbf{a}=a\,\textbf{e}_x \\
\end{cases}
$$

In [ ]:
import numpy as np
from lucifex.mesh import interval_mesh
from lucifex.fem import Constant
from lucifex.fdm import (CN, BE, FE,
    FiniteDifference, FunctionSeries, ConstantSeries,
    advective_timestep)
from lucifex.solver import ibvp , BoundaryConditions
from lucifex.sim import run, Simulation
from lucifex.plt import plot_line, save_figure
from lucifex.utils.py_utils import nested_dict
from lucifex.utils.fenicsx_utils import is_continuous_lagrange
from lucifex.utils.npy_utils import as_index
from lucifex.pde.advection import advection, dg_advection


def create_simulation(
    element: tuple[str, int],
    Lx: float,
    Nx: int,    
    dt: float,
    D_adv: FiniteDifference,
    x0: float,
    sigma: float,
    a: float,
) -> Simulation:
    mesh = interval_mesh(Lx, Nx)
    t = ConstantSeries(mesh, name='t', ics=0.0)
    dt = Constant(mesh, dt, name='dt')
    a = Constant(mesh, (a, ), name='a')
    u = FunctionSeries((mesh, *element), name='u', store=1)
    ics = lambda x: (0.0 + 
        np.cos(np.pi * (x[0] - x0) / sigma) * (x[0] <= x0 + 0.5 * sigma) * (x[0] >= x0 - 0.5 * sigma)
    ) 
    bcs = BoundaryConditions(
        ('dirichlet', lambda x: x[0], 0.0),
    )
    if is_continuous_lagrange(u.function_space):
        u_solver = ibvp(advection, ics, bcs)(u, dt, a, D_adv)
    else:
        u_solver = ibvp(dg_advection, ics)(u, dt, a, D_adv, bcs=bcs)
    return Simulation(u_solver, t, dt)


Lx = 1.0
Nx = 200
h = Lx / Nx
x0 = 0.25 * Lx
sigma = 0.1

a = 1.0
courant = 1.0
dt = advective_timestep(a, h, courant)

elem_opts = [
    ('DP', 0),
    ('DP', 1),
    ('P', 1),
]
D_adv_opts = (FE, BE, CN)
simulations = nested_dict((FiniteDifference, tuple, Simulation))

for elem in elem_opts:
    for D_adv in D_adv_opts:
        simulations[elem][D_adv] = create_simulation(elem, Lx, Nx, dt, D_adv, x0, sigma, a)

n_stop = 30
for elem in elem_opts:
    for D_adv in D_adv_opts:
        run(simulations[elem][D_adv], n_stop) 

In [ ]:
x = np.linspace(0, Lx, num=500)
t_target = dt * 20

for elem in elem_opts:
    fam, deg = elem
    lines, legend_labels = [], []
    legend_title = f'{fam}$_{deg}$\n$C_{{\mathbf{{a}}}}={{{courant}}}$\n\n$\mathcal{{D}}_{{\mathbf{{a}}, u}}$'
    for D_adv in D_adv_opts:
        u = simulations[elem][D_adv]['u']
        time_index = as_index(u.time_series, t_target, condition=lambda x, y: np.isclose(x, y))
        lines.append(u.series[time_index])
        legend_labels.append(f'{D_adv}')
    fig, ax = plot_line(lines, legend_labels, legend_title, x_lims=x, x_label='$x$', y_label=f'$u(t={t_target})$')
    ax.set_ylim(-0.05, 1.1)
    save_figure(f'u(t={t_target})_{fam}{deg}', thumbnail=(elem == ('DP', 0)))(fig)